🌐 [English](dx_app.ipynb) | [한국어](dx_app.ko.ipynb) | [日本語](dx_app.ja.ipynb) | **中文**

# DEEPX Tutorial 02 - DX-APP 使用方法

在第二篇教程中,我们将介绍 DX-APP,并学习如何在 AI 应用中运行编译后的 DXNN 模型。

## 学习目标

完成本教程后,您将能够:

- 了解 DX-APP 为 **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) 中的每一个模型**都提供了可直接运行的 C++ 和 Python 示例代码,并使用这些代码运行您选择的任意模型;
- 描述 DX-APP 是什么,以及它的 C++ 和 Python 示例是如何组织的;
- 只下载教程所需的模型和视频;
- 在图像、视频和摄像头上运行分类、检测、姿态和分割示例;
- 在同步、异步和 PPU 执行方式之间进行选择;
- 启动交互式演示启动器;
- 导出独立的 C++ 或 Python 示例,并在 DX-APP 目录树之外运行它。

## 什么是 DX-APP?

**DX-APP** 是面向 DEEPX NPU 的参考应用。它的核心特点是覆盖面:**[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) 中发布的每一个模型都有与之对应、可直接运行的示例**,同时提供 C++ 和 Python 两个版本,并按任务组织(object detection、classification、face detection、pose、segmentation、depth 等)。您在 Model Zoo 页面选好一个模型,DX-APP 中就已经包含了运行它的代码。DX-APP 帮助开发者快速搭建运行时环境,
并可作为构建和定制自己的 AI 应用的模板。

更多细节请参阅 [dx_app git 仓库](https://github.com/DEEPX-AI/dx_app/blob/main/README.md)中的概述,或从[这里](https://developer.deepx.ai/download/?id=580)下载 DX-APP 用户指南!

> **注意:** 下载用户指南前,必须先登录 https://developer.deepx.ai/。

让我们看看 DX-APP 的文件结构:

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** 演示程序经过优化,可用最少的配置在 DEEPX NPU 上展示预编译模型。
每个演示程序代表一种常见的 AI 任务,可以使用图像、视频或实时摄像头
输入来执行。

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP 仓库布局:**
```shell
dx_app/
├── src/
│   ├── cpp_example/            # C++ end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Model-specific factories, configs, and entry points
│   │   └── common/             # Shared C++ runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   42 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   16 task-specific sync/async runner pairs
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   13 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader
│   │       ├── trackers/       #   Shared IoU tracker
│   │       ├── third_party/    #   Vendored header dependencies
│   │       └── utility/        #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/         # Python end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Four execution/postprocessing variants per model
│   │   └── common/             # Shared Python runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   41 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   SyncRunner, AsyncRunner, arguments, and run utilities
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   14 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader and schema
│   │       ├── trackers/       #   Shared IoU tracker
│   │       └── utility/        #   Labels, preprocessing, profiling, and geometry helpers
│   ├── postprocess/            # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           └── dx_postprocess/ # pybind11 bindings wrapping src/postprocess/
├── config/
│   ├── model_registry.json     # Model registry — single source of truth
│   ├── test_models.conf        # Test model configuration
│   └── README.md               # Config directory documentation
├── scripts/                    # Developer tools, validation, and helper scripts
├── tests/                      # C++, Python, script, helper, and Windows tests
├── assets/                     # Links to downloaded models and videos
├── sample/                     # Bundled images and task-specific sample data
├── bin/                        # C++ executables generated by the build
├── cmake/                      # x86_64 and aarch64 toolchain files
├── extern/                     # External header dependencies
├── build.sh / build.bat        # Linux and Windows build scripts
├── setup.sh / setup.bat        # Model, video, and dependency setup
├── setup_sample_models.sh      # Download the sample model set only
├── setup_sample_videos.sh      # Download the sample video set only
├── run_demo.sh / run_demo.bat  # Interactive demo launcher
├── run_tc.sh                   # Unified test runner for example tests
├── install.sh                  # Dependency and OpenCV installer
└── docs/                       # Detailed documentation
```
---
**DX-APP 同时提供 C++ 和 Python 示例:**

`cpp_example` 和 `python_example` 都遵循相同的"任务优先"结构。

两种语言目前都提供相同的 22 个任务目录:

- 3d_object_detection/
- attribute_recognition/
- classification/
- depth_estimation/
- embedding/
- face_alignment/
- face_detection/
- hand_detection/
- hand_landmark/
- image_denoising/
- image_enhancement/
- instance_segmentation/
- keypoint_detection/
- obb_detection/
- object_detection/
- object_pose_estimation/
- panoptic_driving_perception/
- pose_estimation/
- ppu/
- reid/
- semantic_segmentation/
- super_resolution/

## 前提条件

- 已完成教程 01:使用 `--all` 安装了 DX-Runtime(DX-RT、DX-APP 及其 `venv-dx-runtime` Python 环境),NPU 以 `/dev/dxrt0` 的形式可见。
- DX-APP 示例模型和视频。教程 01 的 3.3 节会下载它们;下面的第 1 节只获取仍然缺失的内容(本教程约需 300 MB 的模型,在全新主机上还需 1.2 GB 的共享示例视频压缩包)。
- 图形桌面会话为可选项:第 2 节的演示单元格会打印帮助文本或以无显示方式运行,第 3 节以无显示方式运行独立示例,并在笔记本中显示保存的图像。
- 不需要 `sudo`。预计约 30 分钟;3.3 节的 C++ 构建需要一到两分钟。

下一个单元格会定位 SDK、检查这些要求并打印状态表。任何标记为 `MISSING` 的项目都会附带提供它的步骤。

## 1. 准备模型和视频

1. 进入 `dx_app` 目录:

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. 使用 `setup.sh` 下载所需的模型和示例视频。该脚本是交互式的(它会询问要下载哪些模型类别),因此需要在终端中运行,而不是在单元格中。打开 **File > New > Terminal** 并运行:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 下一个单元格会用您的实际安装路径打印同样的两条命令。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. 如果您有至少 29 GB 的可用存储空间,请按 **Enter** 下载所有 Model Zoo 模型。

5. 否则,请使用下面的代码单元格只下载本教程需要的模型。教程 01 的 3.3 节已经下载了包含此列表的 24 个模型的超集;如果您已完成该节,`--no-force` 会让单元格跳过所有文件并在几秒内完成。后面的第 3 节会以同样的方式添加它使用的两个 YOLO26 模型。

In [ ]:
# The models used in this tutorial. Tutorial 01 section 3.3 already downloads them;
# this cell only downloads what is still missing (--no-force skips existing files).
SELECTED_MODELS = shlex.join([
    "ResNet50",
    "YoloV7",
    "YoloV7_PPU",
    "yolov8s_pose",
    "yolov8n_seg",
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat "{MODEL_MANIFEST_PATH}" | grep name

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force


6. 验证模型和视频都已按预期下载:

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. 运行演示程序

**DX-APP** 是一组现成的演示应用,展示如何在 DEEPX NPU 上运行编译后的模型,包括分类、检测、分割、姿态估计等多种任务。

用户可以通过探索各种 DX-APP 演示程序来学习并构建应用。

### 2.1 分类
- **bin/resnet50_async** 是运行预训练图像分类模型并输出最高预测类别的 DX-APP 可执行文件。

In [ ]:
%cd {DX_APP_DIR} 

#!ls bin
!./bin/resnet50_async -h
#!./bin/resnet50_async -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg

### 2.2 目标检测
- **bin/yolov\<version>_async** 是 DX_APP 演示二进制文件,它运行基于 YOLO 的目标检测模型,在图像、视频或摄像头中检测目标。

In [ ]:
%cd {DX_APP_DIR} 

# Find other yolo based reference excutable files
!ls ./bin/yolo*

#### 执行模式

DX-APP 提供不同的执行模式,以满足不同的应用需求:

| 模式 | 主要特点 | 推荐用途 |
|---|---|---|
| 同步 | 逐帧顺序处理 | 简单流程和延迟检查 |
| 异步 | 流水线各阶段重叠执行 | 更高的吞吐量 |
| PPU | 卸载受支持的后处理工作 | 更低的主机 CPU 开销 |

- **同步应用**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_sync -h

# Image input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **异步应用**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_async -h

# Image input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (This IP address only works in Korea)
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU 与非 PPU 对比**

下面两个单元格分别用非 PPU 和 PPU 版本的 YOLOv7 模型处理同一段 7 秒的视频片段。当 `SHOW_WINDOW = False`(默认值)时,它们以无显示方式运行(带 `--no-display`),只打印 FPS 摘要,因此笔记本不会阻塞;在桌面会话中设置 `SHOW_WINDOW = True`,即可在 DX-APP 窗口中观看检测结果和 FPS 计数器。比较两行 FPS 输出:PPU 模型把候选框过滤和类别选择移到了 NPU 上。

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


### 2.3 姿态估计
- **bin/yolov8s_pose_sync** 是 DX_APP 演示二进制文件,它运行姿态估计模型,在图像、视频或摄像头中检测人并估计关键点。

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8s_pose_sync -h

# Image input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov

# Camera input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 分割
- **bin/yolov8n_seg_async** 是 DX_APP 演示二进制文件,它运行实例分割模型,为图像、视频或摄像头生成逐像素的类别标签。

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8n_seg_async -h

# Image input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4

# Camera input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 演示脚本

`run_demo.sh` 是面向 C++ 和 Python 演示程序的交互式启动器。

它是交互式的,因此请在终端(**File > New > Terminal**)中运行:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

下一个单元格会用您的实际路径打印同样的命令。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


> ![](assets/sc-run_demo.png)

该启动器在 DX-APP 的各个任务类别中提供了 23 个可直接运行的演示程序。

如果选择 `0` Object Detection,可用的执行选项如下:
> ![](assets/sc-run_demo2.png)

选项 [1, 2] 运行基于 C++ 的应用,选项 [3–6] 运行基于 Python 的应用。

> **注意:** 要运行基于 Python 的应用,请先激活安装了 `dx-engine` 的 Python 虚拟环境。

示例:
> ![](assets/sc-run_demo3.png)

### 2.6 提取用 C++ 和 Python 编写的示例应用

`scripts/dx_tool.sh` 是一个用于管理和导出示例的交互式终端工具。它的 `extract` 菜单会把某个模型的示例连同共享的 `common/` 运行时层一起复制成一个自包含的软件包,可以在 DX-APP 目录树**之外**构建和运行。请在终端(File > New > Terminal)中运行:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

下一个单元格会用您的实际路径打印同样的命令。菜单出现后,请按其下方的步骤操作。

该菜单在后台调用 `scripts/extract_model_package.sh`。第 3 节直接使用该脚本,因为它通过参数接收模型和语言,因此可以在笔记本单元格中运行。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


运行 `dx_tool.sh` 后,您会看到以下菜单。 
- 选择 `3` 查看您感兴趣的可用模型列表。
- 选择 `2` 提取示例代码。
- 选择编程语言。(C++、Python)
- 输入您的应用对应的类别和模型。(例如 `object_detection/yolo26s`)

> ![](assets/sc-extract.png)

几秒钟后,示例代码会生成到 `output` 目录中。

> ![](assets/sc-extract2.png)

## 3. 运行提取出的独立示例

2.6 节以交互方式导出了一个示例。本节在笔记本中用 `scripts/extract_model_package.sh` 完成同样的事情,然后在 **DX-APP 目录树之外**构建并运行导出的软件包,就像在您自己的项目中那样。三个示例覆盖了两种语言以及四种 Python 变体中的两种:

| 示例 | 模型(任务) | 语言与变体 | 展示内容 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26s` | Python,`*_sync.py` | 运行任意 Model Zoo 模型的最简单方式 |
| 3.3 | `object_detection/yolo26s` | C++,`*_sync` | 使用 CMake 构建导出的 C++ 软件包 |
| 3.4 | `pose_estimation/yolo26s_pose` | Python,`*_async_cpp_postprocess.py` | 另一种任务,吞吐量最高的变体 |

每个导出的 Python 示例都会导入 `dx_engine` 和 `dx_postprocess`,它们由 DX-Runtime 安装到 `dx-runtime/venv-dx-runtime` 中。因此这些单元格会显式调用该解释器。导出的示例不包含示例图像,所以单元格会传入 `dx_app/sample/img` 中的一张图像并以无显示方式运行(`--no-display --save`),然后在笔记本中显示保存的结果。

### 3.1 导出软件包

先从脚本自带的帮助开始。它列出了下一个单元格使用的参数:以 `<task>/<model>` 形式指定的模型、`--lang cpp|py|both` 和 `--output-dir`。

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


不加 `--output-dir` 时,脚本会在 `dx_app` 内部就地准备软件包;加上它后,软件包会被复制到 `<dir>/cpp/<task>/<model>` 和 `<dir>/py/<task>/<model>`。下一个单元格把三个软件包导出到本教程的 `workspace/` 目录中,该目录被 git 忽略。

In [ ]:
from IPython.display import Image, display

STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26s yolo26s-pose --no-force

# Export: object detection in both languages, pose estimation in Python.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26s --lang both --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26s_pose --lang py --output-dir "{STANDALONE_DIR}"

!tree -L 3 "{STANDALONE_DIR}"


### 3.2 示例 1:Python 独立运行,目标检测

`yolo26s_sync.py` 是最基本的同步变体:纯 Python 后处理,一次处理一帧。它是最适合首先阅读的变体。DX-APP 中每个 Python 示例的参数都相同:`--model`、一个输入选项(`--image`、`--video`、`--camera`、`--rtsp`),以及用于无显示运行的 `--no-display --save`。

```bash
cd <workspace>/standalone/py/object_detection/yolo26s
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26s"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_sync-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.3 示例 2:C++ 独立运行,目标检测

导出的 C++ 软件包包含 `yolo26s_sync.cpp`、`yolo26s_async.cpp`、共享的 `common/` 层,以及一个链接到已安装 DX-RT 的自动生成的 `CMakeLists.txt`。构建它就是标准的 CMake 流程;没有任何内容回指 DX-APP 目录树。

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26s
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26s_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

请注意,C++ 示例的选项写作 `-m/--model_path` 和 `-i/--image_path`;`./yolo26s_sync --help` 会列出它们。构建需要一到两分钟。

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26s"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26s_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("Yolo26s_sync-*/output.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.4 示例 3:Python 独立运行,姿态估计,异步 + C++ 后处理

每个 Python 示例都有四种变体。文件名会告诉您正在运行的是哪一种:

| 变体 | 后处理 | 线程模型 | 适用场景 |
|---|---|---|---|
| `*_sync.py` | 纯 Python | 同步 | 学习和调试逻辑 |
| `*_async.py` | 纯 Python | 异步 | 性能优化的第一步 |
| `*_sync_cpp_postprocess.py` | C++ 绑定(`dx_postprocess`) | 同步 | CPU 后处理负担较重时 |
| `*_async_cpp_postprocess.py` | C++ 绑定(`dx_postprocess`) | 异步 | 最高 FPS;用于部署的变体 |

本示例切换到另一种任务和最快的变体。命令形式不变:只有目录、脚本名和模型不同。

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26s_pose
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_pose_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26s_pose"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_pose_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_pose_async_cpp_postprocess-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.5 动手尝试:Model Zoo 中的任意模型

这三个示例遵循同一种模式,并且该模式适用于全部 352 个模型:

1. 在 [Model Zoo](https://developer.deepx.ai/modelzoo/) 页面选择一个模型,并用 `./scripts/dx_tool.sh search <keyword>` 或 `./scripts/dx_tool.sh list <category>` 找到它的 DX-APP 名称。
2. 下载它:`./setup.sh --models <name> --no-force`。
3. 导出它:`scripts/extract_model_package.sh <task>/<name> --lang py --output-dir <dir>`。
4. 运行它:`<dir>/py/<task>/<name>/<name>_sync.py --model <file>.dxnn --image <image>`。

下一个单元格会在 `config/model_registry.json` 中查找一个模型,并打印针对它的四条命令。把 `MODEL_NAME` 改成您想尝试的模型,然后在单元格或终端中运行打印出的命令。例如 `scrfd500m`(人脸检测,配合 `sample_face.jpg`)或 `resnet50`(分类,配合 `sample/ILSVRC2012` 中的一张图像)。

In [ ]:
import json

MODEL_NAME = "scrfd500m"   # change me: any model_name from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if e["model_name"] == MODEL_NAME), None)
if entry is None:
    raise KeyError(f"{MODEL_NAME} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {MODEL_NAME}")

task, dxnn = entry["add_model_task"], entry["dxnn_file"]
print(f"{MODEL_NAME}: task={task}, model file={dxnn}\n")
print(f'cd "{DX_APP_DIR}" && bash setup.sh --models {MODEL_NAME} --no-force')
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{MODEL_NAME} --lang py --output-dir "{STANDALONE_DIR}"')
print(f'cd "{STANDALONE_DIR}/py/{task}/{MODEL_NAME}" && "{DX_RUNTIME_PYTHON}" {MODEL_NAME}_sync.py --model "{MODEL_DIR / dxnn}" --image <an image under {SAMPLE_IMG_DIR}> --no-display --save --save-dir "{OUTPUT_DIR}"')


## 4. 故障排除

| 症状 | 看到的内容 | 可能原因 | 解决方法 |
|---|---|---|---|
| 缺少可执行文件 | `./bin/yolov7_async: No such file or directory` | DX-APP 未构建,或构建时未包含该目标 | 在终端中于 `dx_app` 下运行 `./build.sh`(或 `./build.sh --target yolov7_async`) |
| 模型无法加载 | `Failed to load model`,路径位于 `assets/models` 下 | 模型未下载 | 带上模型名运行第 1 节的下载单元格,然后重新运行 |
| 没有输出窗口 | `cannot open display`,或单元格结束时没有出现窗口 | 没有图形会话(SSH 或无显示主机) | 在桌面会话中运行,或使用图像输入(`-i`)并检查保存的结果 |
| 无法打开摄像头 | `Could not open camera 0` | 没有 `/dev/video0`,或用户不在 `video` 组中 | `v4l2-ctl --list-devices`;`sudo usermod -aG video $USER` 然后重新登录 |
| `run_demo.sh` 自动启动了某个演示 | 10 秒后运行了选项 0 | 启动器有 10 秒的输入超时 | 菜单出现后立即输入选项编号 |
| 导出的 Python 示例无法导入 `dx_engine` | `ModuleNotFoundError: No module named 'dx_engine'` | 使用了 Jupyter 的 Python 而不是 DX-Runtime 环境运行 | 像第 3 节的单元格那样使用 `<dx-runtime>/venv-dx-runtime/bin/python` |
| 导出的 C++ 软件包无法完成配置 | `Could not find a package configuration file provided by "dxrt"` | DX-RT 未安装,或安装在自定义前缀下 | 安装 DX-Runtime(教程 01 第 3 节),或在 `cmake` 命令行中添加 `-DCMAKE_PREFIX_PATH=<prefix>` |
| 导出的示例找不到输入 | `Input file not found: sample/img/...` | 软件包不包含示例图像 | 使用 `dx_app/sample/img` 中的文件通过 `--image`(Python)或 `-i`(C++)传入 |

## 5. 总结

### 5.1 已完成的 DX-APP 工作流程

```text
Load SDK paths
       │
       ▼
Prepare selected models and videos
       │
       ▼
Run C++ or Python examples
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Use run_demo.sh or export a standalone example
```

### 5.2 学习成果一览

| 领域 | 您现在能做的事 |
|---|---|
| SDK 路径 | 从 `config.json` 加载 `DX_APP_DIR` 及相关路径 |
| 模型准备 | 只下载练习所需的模型和视频 |
| 示例选择 | 按任务、模型、语言和执行模式查找示例 |
| 执行模式 | 根据目标选择同步、异步或 PPU 执行 |
| 演示启动器 | 使用 `run_demo.sh` 启动已安装的示例 |
| 独立代码 | 使用 `scripts/extract_model_package.sh` 导出任意模型的示例,然后在 DX-APP 之外构建和运行 |
| Model Zoo 覆盖 | 通过 `config/model_registry.json` 或 `dx_tool.sh search` 找到任意 Model Zoo 模型对应的 DX-APP 示例 |

### 5.3 完成清单

- [ ] 确认 DX-RT 能够识别 DEEPX NPU
- [ ] 准备所需的 DX-APP 模型和视频资源
- [ ] 至少运行一个端到端推理示例
- [ ] 比较可用的执行模式
- [ ] 为目标任务找到一个参考实现
- [ ] 打开 `run_demo.sh`
- [ ] 在 DX-APP 目录树之外导出、构建并运行独立的 Python 和 C++ 示例
- [ ] 运行一个您自己选择的 Model Zoo 模型

> **下一步:** 继续学习教程 03,改造一个已训练的模型,将其编译为 DXNN,并集成到 DX-APP 运行时配置中。